# PG-AMF Bearing Fault Diagnosis — Experiments Notebook

This notebook is an **interactive companion** to the modular Python pipeline in `src/`.
Use it for quick experiments, visualisations, and sanity checks.

For production training / evaluation, run the scripts directly:
```
python scripts/train.py
python scripts/evaluate.py
python scripts/generate_figures.py
```

## 1 · Setup

In [ ]:
import sys
sys.path.insert(0, '..')   # so `import src` works from the notebooks/ folder

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils import load_config, set_seed, get_device, compute_char_freqs, setup_logging

setup_logging()
cfg    = load_config('../configs/default.yaml',
                     '../configs/data.yaml',
                     '../configs/model.yaml',
                     '../configs/train.yaml')
seed   = cfg['seed']
device = get_device(cfg['device'])
set_seed(seed)

print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')

## 2 · Load Data

In [ ]:
from src.dataset import BearingDataLoader

# Change data_root to point to your XJTU_Gearbox folder
# If the folder doesn't exist, synthetic data is used automatically
loader = BearingDataLoader(
    data_root  = cfg['paths']['data_root'],
    fault_types= cfg['fault_types'],
    fault_labels=cfg['fault_labels'],
    fs         = cfg['data']['signal']['fs'],
    seed       = seed,
)
data_dict = loader.load_all()
df_raw    = loader.to_dataframe(data_dict)
df_raw[['fault_name', 'n_samples', 'synthetic']]

## 3 · Preprocessing & Dataset

In [ ]:
from src.preprocessing import build_dataset, split_dataset, extract_stat_features, fit_scaler

sig = cfg['data']['signal']
X_all, y_all = build_dataset(df_raw, sig['seg_len'], sig['hop'],
                              sig['min_segs'], sig['max_segs'],
                              sig['do_mean_removal'])

splits = cfg['data']['splits']
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset(
    X_all, y_all, splits['val_split'], splits['test_split'], seed
)

print(f'X_all  : {X_all.shape}   (N, 2_channels, seg_len)')
print(f'Train  : {X_train.shape[0]}   Val: {X_val.shape[0]}   Test: {X_test.shape[0]}')
print(f'Class balance: {np.bincount(y_all).tolist()}')

## 4 · Quick Signal Plot

In [ ]:
fault_colors = cfg['fault_colors']
fault_names  = list(cfg['fault_labels'].values())
t_axis = np.arange(sig['seg_len']) / sig['fs'] * 1000  # ms

fig, axes = plt.subplots(len(fault_names), 1, figsize=(10, 8), sharex=True)
for i, (_, row) in enumerate(df_raw.iterrows()):
    axes[i].plot(t_axis, row['ch1'][:sig['seg_len']], color=fault_colors[i], lw=0.7)
    axes[i].set_ylabel(row['fault_name'], color=fault_colors[i], fontsize=9)
axes[-1].set_xlabel('Time [ms]')
fig.suptitle('Ch1 Waveforms — All Fault Classes', fontweight='bold')
plt.tight_layout()
plt.show()

## 5 · Train (Quick Smoke-Test — 5 Epochs)

In [ ]:
from src.model import MLPBaseline, PGAMFClassifier
from src.trainer import make_loaders, train_basic, train_smooth, evaluate_model

F_stat_train = fit_scaler(extract_stat_features(X_train)).transform(extract_stat_features(X_train))
scaler = fit_scaler(extract_stat_features(X_train))
F_stat_train = scaler.transform(extract_stat_features(X_train))
F_stat_val   = scaler.transform(extract_stat_features(X_val))
F_stat_test  = scaler.transform(extract_stat_features(X_test))
n_stat_feat  = F_stat_train.shape[1]
n_classes    = len(cfg['fault_types'])
mc = cfg['model']

dl_stat_tr, dl_stat_v, dl_stat_te = make_loaders(
    F_stat_train, y_train, F_stat_val, y_val, F_stat_test, y_test,
    batch_size=cfg['train']['batch_size'], device=device)

model_stat = MLPBaseline(n_stat_feat, mc['baseline_mlp']['hidden_size'], n_classes).to(device)
hist = train_basic(model_stat, dl_stat_tr, dl_stat_v, name='Stat MLP (smoke)', epochs=5, patience=5)
acc, _, _ = evaluate_model(model_stat, dl_stat_te)
print(f'\nTest accuracy (5 epochs): {acc*100:.1f}%')

## 6 · Learned α Exponents (after full training)

In [ ]:
# Load a trained PG-AMF checkpoint (if available)
import torch
from pathlib import Path

pg = mc['pgamf']
model_pgamf = PGAMFClassifier(
    n_classes=n_classes, F=pg['F'], n_channels=pg['n_channels'],
    hidden=pg['hidden_size'], lambda_div=pg['lambda_div'],
    lambda_compact=pg['lambda_compact'],
    alpha_min=pg['alpha_min'], alpha_max=pg['alpha_max'],
).to(device)

ckpt = Path('../outputs/checkpoints/pgamf_best.pt')
if ckpt.exists():
    model_pgamf.load_state_dict(torch.load(ckpt, map_location=device))
    print('Checkpoint loaded.')
else:
    print('No checkpoint found — using random weights.')

with torch.no_grad():
    alphas = model_pgamf.pgamf.alphas.cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 2.5))
for ch, clr, mk in zip(range(2), ['#2196F3', '#F44336'], ['o', 's']):
    a_sorted = np.sort(alphas[ch])
    ax.scatter(range(pg['F']), a_sorted, color=clr, marker=mk, s=60,
               edgecolors='black', lw=0.5, label=f'Ch{ch+1}')
    ax.plot(range(pg['F']), a_sorted, color=clr, ls='--', lw=0.8, alpha=0.5)
ax.axhline(2.0, color='g', ls=':', lw=1.0, label='RMS (α=2)')
ax.axhline(4.0, color='m', ls=':', lw=1.0, label='Kurtosis~(α=4)')
ax.set_xlabel('Moment Index')
ax.set_ylabel(r'Learned $\alpha_i$')
ax.set_title('Learned PG-AMF Exponents')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()